# RLHF Alignment: How Do We Make an LLM Speak Like a Helpful Human?

> **Background**: you trained an LLM. It is smart and knows a lot. But if you ask for something harmful, it may answer anyway. That is not acceptable.
>
> Alignment goal: make the model **Helpful, Honest, and Harmless (3H)**.
>
> Goal for this part: **walk through SFT -> Reward Model -> PPO -> DPO, explain why each step exists, and show the core math with small hand-worked examples.**

Core tension:
**a language model is trained to predict the next token, but we want it to produce helpful answers.** Those are not the same objective.

RLHF (Reinforcement Learning from Human Feedback) bridges the gap by using human preference signals.

Note: this part is about classic alignment (human preference). It is different from "RL for reasoning" which targets better reasoning quality.


In [ ]:
import numpy as np
import math

np.random.seed(42)

### 1. Why alignment? Feel the difference with two answers

Pretraining teaches a model to continue text patterns, not to be helpful.

Example:

```
User: I've been feeling terrible lately.

Unaligned base model (pattern completion):
  It continues a dramatic story it saw online...
  The pattern may be fluent, but it is not the right response.

Aligned model (Helpful + Harmless):
  It responds with empathy, encourages seeking real help, and asks gentle follow-ups.
```

**Pretraining teaches the model to talk. Alignment teaches the model to talk like a responsible assistant.**

3H principles:
- Helpful: answer the question, stay on topic
- Honest: say "I don't know" when uncertain; avoid hallucination
- Harmless: refuse harmful requests


### 2. The alignment map (full pipeline)

The whole chain has stages:

```
Base model (pretrained, only continuation)
     |
     v  Stage 1: SFT
+------------------------------+
| Supervised fine-tuning on    |
| high-quality conversations    |
| teaches chat format + basic   |
| instruction following         |
+--------------+---------------+
               v
          SFT model
      (can chat, but cannot
       reliably rank good vs bad)
               |
        +------+------+
        |             |
        v             v
 Stage 2a: RLHF    Stage 2b: DPO
 (classic route)    (simpler route)
   1) collect prefs    1) collect prefs
   2) train RM         2) optimize prefs directly
   3) PPO training         (skip RM + PPO)
        \             /
         \           /
          v         v
        Aligned model
```

Next we go stage by stage.


---

### 3. Stage 1: SFT (teach the model to chat)

#### 3.1 What SFT does

SFT is supervised learning on high-quality dialogue data, e.g.

```
<User> What is the capital of France?
<Assistant> The capital of France is Paris.
<User> Why Paris?
<Assistant> Paris is the largest city and the political center of France...
```

SFT is still cross-entropy next-token prediction. The only difference is that data shifts from raw internet text to instruction/dialogue.

After SFT, the model learns:
1. chat format (turn-taking)
2. basic instruction-following patterns
3. surface features of "good answers"

#### 3.2 The key limitation

For problems with a single correct answer, SFT is often enough.
But for open-ended tasks (e.g. "write a resignation letter"), there is no single gold answer.
Which answer is *better*?
SFT does not provide that signal.

That motivates the next stage: collect human preferences.


---

### 4. Stage 2: Reward Model (teach "what is good")

#### 4.1 What preference data looks like

For the same prompt, annotators compare two answers and choose the better one:

```
Prompt: Write a resignation letter.

Answer A (chosen):
  polite, clear, professional...

Answer B (rejected):
  rude and unhelpful...
```

So one preference example is `(prompt, chosen, rejected)`.

#### 4.2 Reward model objective

A reward model outputs a scalar score `r(x)` for a response.
We want `r(chosen) > r(rejected)`.
A common loss is a logistic ranking loss:

`L_RM = -log(sigmoid(r_chosen - r_rejected))`

Intuition: make the chosen score higher than the rejected score.


In [ ]:
# === Reward Model Loss manual ===
print("=== Reward Model loss ===")
print()
print(" : L = -log( σ(r_chosen - r_rejected) )")
print(" σ(x) = 1/(1+e^(-x)) sigmoid")
print()

def sigmoid(x):
    return 1 / (1 + math.exp(-x))

def reward_loss(r_chosen, r_rejected):
    diff = r_chosen - r_rejected
    prob = sigmoid(diff)
    loss = -math.log(max(prob, 1e-10))
    return loss, diff, prob

#  
cases = [
    (" : chosen >> rejected", 8.0, 2.0),
    (" : chosen ", 6.0, 5.0),
    (" : chosen < rejected", 3.0, 7.0),
    (" : chosen << rejected", 1.0, 9.0),
]

print(f"{' ':<25s} {'r_c':>6s} {'r_r':>6s} {'diff':>8s} {'σ(diff)':>10s} {'loss':>10s}")
print("-" * 70)

for desc, r_c, r_r in cases:
    loss, diff, prob = reward_loss(r_c, r_r)
    print(f"{desc:<25s} {r_c:>6.1f} {r_r:>6.1f} {diff:>8.1f} {prob:>10.4f} {loss:>10.4f}")

print()
print(" :")
print(" • σ(diff) 「chosen 」")
print(" • r_chosen >> r_rejected: σ≈1, loss≈0 -> RM , ")
print(" • r_chosen << r_rejected: σ≈0, loss -> RM , ")
print()
print(" 「 」 ：")
print(" =1 (chosen ), =σ(r_c - r_r),loss=cross_entropy")

---

### 5. Stage 3: PPO (optimize the policy under reward + constraints)

PPO uses the reward model as the training signal.

Core idea:
- sample responses from the current policy
- score them with the reward model
- update the policy to increase expected reward
- but constrain updates to avoid instability and reward hacking

A common PPO-style objective:

```
L_PPO = -E[ min(ratio * A, clip(ratio, 1-eps, 1+eps) * A) ]
        + beta * KL(pi_theta || pi_SFT)

ratio = pi_theta(token|prompt) / pi_old(token|prompt)
A     = advantage = RM_score - baseline
KL    = penalty to stay close to the SFT reference policy
beta  = KL weight (often 0.01-0.1)
```

KL penalty is important: without it, the model may find shortcuts that exploit RM weaknesses (reward hacking).


In [ ]:
# === PPO clip manual ===
print("=== PPO Clip ===")
print()

def ppo_loss(ratio, advantage, epsilon=0.2):
    """ PPO surrogate loss ratio = π_new / π_old ( ) advantage = RM_score - baseline ( ) """
    #  clip loss
    unclipped = ratio * advantage
    #  clip loss( ratio [0.8, 1.2])
    clipped = np.clip(ratio, 1-epsilon, 1+epsilon) * advantage
    # PPO ( advantage>0 min, advantage<0 max)
    #  -min(unclipped, clipped) surrogate
    return -min(unclipped, clipped)


ratios = [0.5, 0.7, 0.9, 1.0, 1.1, 1.3, 1.5, 2.0]

print(" advantage > 0( , )：")
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {' '}")
print("-" * 60)
for r in ratios:
    unclipped = r * 1.0
    clipped = min(r, 1.2) * 1.0
    loss = -min(unclipped, clipped)
    note = ""
    if r > 1.2:
        note = "← clip! "
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print(" advantage < 0( , )：")
print(f"{'ratio':>8s} {'unclipped':>12s} {'clipped':>12s} {'loss':>10s} {' '}")
print("-" * 60)
for r in ratios:
    unclipped = r * (-1.0)
    clipped = max(r, 0.8) * (-1.0)
    loss = -min(unclipped, clipped)
    note = ""
    if r < 0.8:
        note = "← clip! "
    print(f"{r:>8.2f} {unclipped:>12.2f} {clipped:>12.2f} {loss:>10.2f} {note}")

print()
print(" ：")
print(" • advantage>0( token): PPO ratio, ≤1.2")
print(" • advantage<0( token): PPO ratio, ≥0.8")
print(" • clip = -> , training ")

---

### 6. DPO: skip RM and PPO, optimize preferences directly

#### 6.1 Why DPO?

RLHF is powerful but engineering-heavy:
- train a separate reward model
- PPO training runs multiple models (actor/ref/RM/critic)
- many hyperparameters
- unstable optimization

DPO (Direct Preference Optimization, 2023) proposes a simpler alternative.

#### 6.2 DPO intuition

DPO yields a preference loss of the form:

```
L_DPO = -log( sigma(
    beta * log( pi_theta(chosen) / pi_ref(chosen) )
  - beta * log( pi_theta(rejected) / pi_ref(rejected) )
))
```

Interpretation:
- increase probability of chosen answers -> loss down
- increase probability of rejected answers -> loss up
- reference policy in the denominator plays a role similar to KL regularization
- beta controls how aggressive the update is


### 7. RLHF vs DPO (comparison)

| Dimension | RLHF (PPO) | DPO |
|:---|:---|:---|
| #models during training | 4 (actor, ref, RM, critic) | 2 (train, ref) |
| training style | online (generate every step) | offline (fixed preference pairs) |
| stability | worse (PPO is hard) | better (supervised-style) |
| reward hacking risk | higher (fixed RM target) | lower (no RM to exploit) |
| ceiling | potentially higher | limited by preference data |
| debugging | harder | easier |

Rule of thumb:
- small teams / fast iteration -> DPO
- large teams / max performance -> RLHF


In [ ]:
# === DPO Loss manual ===
print("=== DPO Loss ===")
print()

def dpo_loss(logp_chosen, logp_rejected, logp_ref_chosen, logp_ref_rejected, beta=0.5):
    """ DPO loss logp = log(probability of generating this response) ( ) """
    # chosen ref 
    chosen_improvement = logp_chosen - logp_ref_chosen
    # rejected ref ( )
    rejected_improvement = logp_rejected - logp_ref_rejected
    
    #  ：chosen rejected 
    diff = beta * (chosen_improvement - rejected_improvement)
    
    # sigmoid -log
    prob = 1 / (1 + math.exp(-diff))
    loss = -math.log(max(prob, 1e-10))
    
    return loss, chosen_improvement, rejected_improvement, diff, prob


scenarios = [
    (" : chosen↑, rejected↓",    -1.0, -5.0, -3.0, -3.0),
    (" : chosen↑, rejected ↑ ", -1.0, -2.5, -3.0, -3.0),
    (" : chosen↓, rejected↑",     -5.0, -1.0, -3.0, -3.0),
    (" chosen ",            -0.5, -2.0, -3.0, -3.0),
    (" chosen ",            -4.5, -6.0, -3.0, -3.0),
]

print(f"{' ':<28s} {'chosen_imp':>10s} {'rej_imp':>10s} {'diff':>10s} {'loss':>10s}")
print("-" * 74)

for desc, lc, lr, ref_c, ref_r in scenarios:
    loss, ci, ri, diff, prob = dpo_loss(lc, lr, ref_c, ref_r)
    print(f"{desc:<28s} {ci:>+10.2f} {ri:>+10.2f} {diff:>10.2f} {loss:>10.4f}")

print()
print(" : DPO , chosen rejected ")
print(" chosen + rejected -> diff -> loss ")
print(" chosen rejected ( ),")
print(" chosen rejected -> diff > 0 -> loss ")

### 8. A complete alignment recipe (example: LLaMA 2)

```
Stage 1: Pretraining
  base model on ~2T tokens

Stage 2: SFT
  ~27k high-quality human conversations
  train a few epochs

Stage 3: Preference data
  ~1M+ (prompt, chosen, rejected) comparisons

Stage 4a: Reward model
  initialize from SFT, train to score

Stage 4b: PPO
  use RM score -> PPO optimize
  multiple iterations

Result: chat-aligned model
```


### 9. Limitations and side effects

RLHF/DPO improves safety, but can introduce new issues:

1. over-refusal (benign questions get refused)
2. sycophancy (agreeing with the user even when wrong)
3. style homogenization (everything sounds the same)
4. knowledge loss (forgetting some pretraining knowledge)

These are active research problems.

Also, for non-chat tasks like code completion or translation, alignment is not always the main lever; the definition of "good" may come from the downstream task metrics.


---

## RLHF Checklist

1. [ok] Why alignment: pretraining teaches "talking"; alignment teaches "helpful talking"
2. [ok] 3H: Helpful + Honest + Harmless
3. [ok] SFT: learn chat format; cannot rank good vs bad well
4. [ok] Preference data: (prompt, chosen, rejected)
5. [ok] Reward model loss: `-log(sigmoid(r_chosen - r_rejected))`
6. [ok] PPO clip: ratio = pi_new/pi_old, clipped to control step size
7. [ok] KL penalty: prevents reward hacking / drifting too far
8. [ok] DPO: skip RM+PPO, optimize preference pairs directly
9. [ok] RLHF vs DPO: RLHF higher ceiling but complex; DPO simpler and strong

One-sentence summary: alignment = SFT baseline (learn to chat) -> define goodness (RM or preference loss) -> optimize policy (PPO or DPO). DPO makes this accessible to smaller teams.


(This notebook intentionally avoids giving operational harmful instructions; examples are illustrative only.)
